### This notebook will get the Airline Data from the Azure Blob Storage and stores in the Table in the Databricks Environment

### Step 1: Set the data location and type

To get started, we need to set the location and type of the file.

In [ ]:
%python
adls_storage_name = "airlinedatasetproject"
adls_storage_access_key = "<YOUR_STORAGE_ACCOUNT_KEY>"

In [ ]:
%python
file_location = "wasbs://airlines-data@airlinedatasetproject.blob.core.windows.net/raw-data"
file_type = "csv"

In [ ]:
%python
spark.conf.set(
  "fs.azure.account.key."+adls_storage_name+".blob.core.windows.net", adls_storage_access_key)

### Step 2: Read all the airline data present under the folder airport

Now that we have specified our file metadata, we can create a DataFrame. Notice that we use an *option* to specify that we want to infer the schema from the file. We can also explicitly set this to a particular schema if we have one already.

First, let's create a DataFrame in Python.

In [ ]:
%python
df_air = spark.read.format(file_type).option("inferSchema", "true").option("header", "true").load(file_location)

### Step 3: Query the data

Now that we have created our DataFrame, we can query it. For instance, you can identify particular columns to select and display.

In [ ]:
%python
display(df_air)

### Step 4: Create a temporary view or table for analysing the data

If you want to query this data as a table, you can simply register it as a *view* or a table.

In [ ]:
%python
df_air.createOrReplaceTempView("airlineTemp")

We can query this view using Spark SQL. For instance, we can perform a simple aggregation. Notice how we can use `%sql` to query the view from SQL.

In [ ]:
%sql
SELECT * FROM airlineTemp WHERE TAIL_NUM = 'N743SW'

In [ ]:
%sql
DROP TABLE IF EXISTS airline_cluster.airline.airlineReporting

### Step 5: Create a Permanent Table

Since this table is registered as a temp view, it will be available only to this notebook. If you'd like other users to be able to query this table, we have to create a permanent table from the DataFrame as below.

In [ ]:
spark.sql("USE CATALOG airline_cluster")
spark.sql("CREATE SCHEMA IF NOT EXISTS airline_cluster.airline")
spark.sql("USE SCHEMA airline")
spark.sql("SELECT current_catalog(), current_schema()").show()

df_air.write.format("delta").mode("overwrite").saveAsTable("airlineReporting")


In [ ]:
%sql
Select * from airlineReporting
WHERE TAIL_NUM = 'N743SW'

In [ ]:
%sql
Select count(1) from airlineReporting

#### This table will persist across cluster restarts and allow various users across different notebooks to query this data.